# 🔎 Retrieval-Augmented Generation (RAG) 📚

RAG combines **information retrieval** 📚 with **LLM generation** 🧠.  
Instead of relying only on the model’s memory, it **fetches relevant data** from a database or documents **just in time**.

This makes responses **accurate, grounded, and up-to-date**.

---

## 🎯 Learning Objectives
- Understand **document chunking & overlap strategies**
- Explore **dense vs sparse retrieval**
- Learn about **vector databases & similarity search**
- Discover **cross-encoder reranking**
- Apply **context injection & templating**
- Evaluate RAG using **RAGAS, recall@k & grounding**

## 1. Why RAG? 🤔

LLMs are powerful but have **limited memory and outdated training data**.  
RAG lets them:
- Fetch **real-time info** 🌍
- Reduce **hallucinations** 🫠
- Work on **private datasets** 🗄️


## 2. Document Chunking & Overlap ✂️

Documents are too big for LLMs to read at once.  
We **split them into small pieces (chunks)** with slight overlap to preserve context.

Example: 500 tokens per chunk with 50 tokens overlap.


In [16]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.document_loaders import TextLoader

# Load the document (ensure the file path is correct)
loader = TextLoader("data/sample.txt")
documents = loader.load()

# Create a text splitter with overlap configuration
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,  # Set chunk size to 100 characters for better visibility of overlap
    chunk_overlap=150,  # Set overlap to 50 characters
)

# Split documents into chunks
chunks = text_splitter.split_documents(documents)

# Print the chunks with clear overlap demonstration
for i, chunk in enumerate(chunks[:-1]):  # Skip the last chunk to prevent out-of-range error
    print(f"Chunk {i+1}:")
    print(chunk.page_content)
    print("\n---\n")


Chunk 1:
One day, an old man was walking along a beach that was littered with thousands of starfish that had been washed ashore by the tide. As he walked, he saw a young boy in the distance, picking something up and gently throwing it into the ocean.
As the man approached, he called out, “Good morning! May I ask what you are doing?”

---

Chunk 2:
As the man approached, he called out, “Good morning! May I ask what you are doing?”
The boy paused, looked up, and replied, “I’m throwing starfish back into the ocean. The tide has washed them up and they can’t return to the sea by themselves. If I don’t throw them back, they’ll die.”

---

Chunk 3:
The old man replied, “But there must be tens of thousands of starfish on this beach. I’m afraid you won’t really be able to make much of a difference.”
The boy bent down, picked up another starfish, and threw it as far as he could into the ocean. Then he turned, smiled, and said, “I made a difference to that one.”

---



## 3. Dense Embeddings 🌐

Chunks are transformed into **high-dimensional vectors**  
representing their meaning using embedding models like:
- **E5**
- **BGE**
- **MiniLM**

Each chunk is converted into a **dense vector** (high-dimensional number list).  
Dense = captures meaning & semantics, not just words.
Dense embeddings = semantic understanding (not just keyword matching).


In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

# Load a pre-trained MiniLM model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Example text chunks
chunk1 = "Neural networks are computational models inspired by the brain's structure."
chunk2 = "Machine learning algorithms, including deep learning, require large datasets to train models."
chunk3 = "india is country in asia"

# Generate dense embeddings for each chunk
embedding1 = model.encode(chunk1)
embedding2 = model.encode(chunk2)
embedding3 = model.encode(chunk3)

# Print out the vectors
print("Embedding for Chunk 1:")
print(embedding1[:50])
print("\nEmbedding for Chunk 2:")
print(embedding2)

Embedding for Chunk 1:
[-0.0525718  -0.09527728  0.03029311 -0.00585646 -0.02443227  0.00106506
 -0.03892764 -0.00348536  0.10915745 -0.02609401 -0.02114871  0.01485094
  0.03116974 -0.00541098 -0.05564336 -0.06907054 -0.08539793  0.01945207
 -0.06242407  0.00588588  0.0141584   0.02437687 -0.0593435  -0.00257715
 -0.03734461  0.08393001  0.00694768  0.01841992 -0.00929384  0.02469327
  0.11749815 -0.05867224 -0.00762798  0.04334841 -0.04633474  0.04331664
 -0.06198166 -0.03874096 -0.00281439  0.02193852 -0.03099974  0.00526311
 -0.00519046  0.03654814  0.140013    0.05900377 -0.01684062 -0.06184499
 -0.06060772 -0.0164155 ]

Embedding for Chunk 2:
[ 2.84525286e-03 -6.06521368e-02  5.59094138e-02  1.41367642e-03
  2.85753980e-02 -4.83164266e-02 -1.71113491e-01 -4.71944995e-02
 -3.31989378e-02 -3.65512557e-02 -4.76886854e-02  5.21095656e-03
 -1.40983984e-02  1.09480526e-02 -4.75357138e-02  1.57240890e-02
  6.23587845e-03  2.96441559e-02 -1.10100217e-01 -4.00550142e-02
  2.02732868e-02  

/Users/kaps/Developer/thinkOpenAi/.venv/lib/python3.11/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


## 4. Vector Databases 🗄️

We store embeddings in specialized databases for **fast similarity search**:
- **Chroma**
- **FAISS**
- **Qdrant**
- **Weaviate**

These make retrieving relevant context lightning-fast ⚡.


In [33]:
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

# Step 1: Initialize the model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Step 2: Create example sentences
sentences = [
    "The cat sat on the mat.",
    "A dog barked loudly.",
    "She loves programming in Python.",
    "I am learning machine learning.",
    "The sun is shining brightly."
]

# Step 3: Create embeddings for these sentences
embeddings = model.encode(sentences)

# Step 4: Create a FAISS index and add the embeddings
dimension = embeddings.shape[1]  # 768 for 'all-MiniLM-L6-v2'
index = faiss.IndexFlatL2(dimension)  # L2 distance-based index

# Convert embeddings to float32 (FAISS expects this format)
embeddings = np.array(embeddings, dtype=np.float32)

# Add the embeddings to the index
index.add(embeddings)

# Step 5: Query the index (e.g., find sentences similar to "I want a pet")
query = "I want a pet"
# query = "I want to learning"
query_embedding = model.encode([query])

# Perform a search with k=3 (top 3 results)
k = 3
distances, indices = index.search(np.array(query_embedding, dtype=np.float32), k)

# Display the results
print(f"Query: '{query}'")
print("\nMost similar sentences:")
for i, idx in enumerate(indices[0]):
    print(f"{i+1}. {sentences[idx]} (Distance: {distances[0][i]:.4f})")


Query: 'I want a pet'

Most similar sentences:
1. A dog barked loudly. (Distance: 1.3972)
2. The cat sat on the mat. (Distance: 1.4740)
3. She loves programming in Python. (Distance: 1.5750)


## 5. Similarity Search (Cosine, Dot-Product) 🧲

When a query comes in:
1. Convert it into an embedding.
2. Compare it to stored vectors using:
   - **Cosine similarity** (most common)
   - **Dot product**
3. Fetch the top-matching chunks.


Cosine Similarity:
It measures the angle between two vectors (embeddings). A cosine similarity of 1 means the vectors are identical, while a value closer to -1 means they are very different in terms of their direction.

In [35]:
# Example text chunks
chunk1 = "Neural networks are computational models inspired by the brain's structure."
chunk2 = "Machine learning algorithms, including deep learning, require large datasets to train models."
chunk3 = "india is country in asia"

# Generate dense embeddings for each chunk
embedding1 = model.encode(chunk1)
embedding2 = model.encode(chunk2)
embedding3 = model.encode(chunk3)

# Calculate cosine similarity between the two chunks (semantic similarity)
from sklearn.metrics.pairwise import cosine_similarity

similarity = cosine_similarity([embedding1], [embedding2])
print("\nCosine Similarity:", similarity[0][0])

similarity = cosine_similarity([embedding1], [embedding3])
print("\nCosine Similarity:", similarity[0][0])


Cosine Similarity: 0.4492445

Cosine Similarity: 0.047278326


/Users/kaps/Developer/thinkOpenAi/.venv/lib/python3.11/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


Dot Product:
It computes the similarity between two vectors based on their projection. A larger dot product generally indicates higher similarity. It directly depends on both the direction and magnitude of the vectors.

In [38]:
# Calculate dot product between the two chunks
dot_product_1_2 = np.dot(embedding1, embedding2)
dot_product_1_3 = np.dot(embedding1, embedding3)

# Print Dot Product Results
print("\nDot Product between chunk1 and chunk2:", dot_product_1_2)
print("\nDot Product between chunk1 and chunk3:", dot_product_1_3)


Dot Product between chunk1 and chunk2: 0.4492445

Dot Product between chunk1 and chunk3: 0.047278326


## 6. Sparse Retrieval (BM25, TF-IDF) 📄

An older but still useful approach using **keywords & term frequencies**.  
- **BM25** → Ranks documents by keyword relevance.
- **TF-IDF** → Weights rare but important terms.

Works well for **hybrid search** (dense + sparse).


## 7. Cross-Encoder Reranking 🥈

After fetching chunks, we can **re-rank them**  
using a **more accurate but slower cross-encoder**.

This ensures **only the best matches** go into the LLM.


**Cross-Encoder Reranking** is a technique used to improve the relevance of search results by **re-ranking** them based on how well they match a given query. The key idea is to use a more accurate but slower model, called a **cross-encoder**, after fetching initial results. 🔄

### **How It Works:**

1. **Initial Search** 🔍: You first fetch a few candidate answers or text chunks based on the query.
2. **Reranking with Cross-Encoder** ⚖️: A **cross-encoder** model evaluates each candidate chunk in the context of the query. It compares both the query and the text chunk together to calculate a relevance score.
3. **Final Ranking** 🏆: Based on the scores, the chunks are reordered, with the most relevant results ranked higher.

### **Example**:

Let’s say a student asks:
**"What’s the difference between machine learning and deep learning?"** 🤖

Initially, the system might return these 3 chunks:

1. **Chunk 1**: "Machine learning uses algorithms to find patterns in data."
2. **Chunk 2**: "Deep learning is a subset of machine learning using neural networks."
3. **Chunk 3**: "Machine learning is used in finance, healthcare, and marketing."

While **Chunk 2** is the most relevant, the system may rank them incorrectly at first. A **cross-encoder** model then re-ranks them. It compares the query with each chunk and finds that **Chunk 2** clearly explains the difference, so it ranks it highest. 🥇

### **Why Cross-Encoders?**

* **Accuracy** 🎯: They analyze the query and text together, making them more accurate at determining relevance.
* **Efficiency** ⚡: They're used only to rerank a small set of initial results, so the search process is still fast.

### **In Short**:

Cross-encoders ensure that only the **best possible answers** are presented to the user, even if it takes a bit more time, improving the quality of search results. 💡


In [26]:
from sentence_transformers import SentenceTransformer
from sentence_transformers  import CrossEncoder
import numpy as np

# Load cross-encoder model (this will perform query-document reranking)
model = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

# Example query and retrieved documents
query = "What is the capital of France?"
retrieved_docs = [
    "Paris is the capital of France.",
    "i am computer programmer",
    "The Eiffel Tower is located in Paris.",
    "France is a country in Europe."
]

# Create query-document pairs (query + each retrieved document)
pairs = [(query, doc) for doc in retrieved_docs]

# Use the cross-encoder to get relevance scores for each pair
scores = model.predict(pairs)

print(scores)

# Sort documents based on the relevance score in descending order
sorted_indices = np.argsort(scores)[::-1]  # Sort indices based on scores in descending order

print(sorted_indices[:2])

# Get the top-N documents based on reranking
top_k_docs = [retrieved_docs[i] for i in sorted_indices[:2]]  # Top 2 most relevant documents

# Display reranked documents
print("query is - What is the capital of France")
print("Top-K Reranked Documents:")
for doc in top_k_docs:
    print(doc)


[  8.500708  -10.992187   -6.6122355  -1.1091869]
[0 3]
query is - What is the capital of France
Top-K Reranked Documents:
Paris is the capital of France.
France is a country in Europe.


## 7.1 Context Injection & Templating 🧩

### What is Context Injection and Templating?

**Context Injection** means adding relevant pieces of information to a prompt, guiding the model to generate more accurate and focused responses. **Templating** is creating a structured format for how this information should be presented to the AI. Together, they make AI responses more useful and relevant.

### Why is it Important?

Imagine you ask an AI, “What is machine learning?” without context. It may give a generic answer. But if you provide specific context, like “Explain machine learning algorithms,” the model will generate a much more focused response. That’s the power of **context injection** and **templating**! 🎯

### How Does it Work?

1. **Extracting Relevant Information** 📄:

   * From a list of sentences or data, we first identify the most relevant piece. For example, you have a list of sentences about AI and machine learning. We pick the sentence most related to the topic you're interested in.

2. **Creating a Template** 📝:

   * A template is a pre-defined structure, like:

     ```
     "Please write a 300-word explanation about the topic: '{topic}'."
     ```

3. **Injecting Context into the Template** 💡:

   * The relevant sentence (e.g., about machine learning) is inserted into the template. So, your prompt now becomes:

     ```
     "Please write a 300-word explanation about the topic: 'machine learning algorithms'."
     ```

4. **Generating Content** 🚀:

   * This context-injected prompt is sent to the AI model, guiding it to generate a more relevant response.

### Example:

If the topic is **"machine learning algorithms"**, the model might generate:

```
"Machine learning algorithms allow computers to learn from data without being explicitly programmed..."
```

### Why Does This Matter? 🤔

Context injection ensures that the model understands the specific question or topic. It helps in getting more precise, useful responses, saving time and improving communication with AI.


In [ ]:
import openai
import numpy as np
from sentence_transformers import SentenceTransformer

# Step 1: Initialize the sentence transformer model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Example list of sentences
sentences = [
    "The cat sat on the mat.",
    "I am interested in learning about machine learning algorithms.",
    "Python is a great programming language for data science.",
    "Quantum computing is a fascinating and rapidly evolving field.",
    "Machine learning algorithms can help us make data-driven decisions."
]

# Step 2: Define the keyword to search for
keyword = "machine learning"  # For example, you can change this to any keyword

# Step 3: Encode the sentences and the keyword
sentence_embeddings = model.encode(sentences)
keyword_embedding = model.encode([keyword])

# Step 4: Compute the similarity between the keyword and all sentences
# Using cosine similarity
cosine_similarities = np.dot(sentence_embeddings, keyword_embedding.T) / (
    np.linalg.norm(sentence_embeddings, axis=1) * np.linalg.norm(keyword_embedding))

# Step 5: Find the most relevant sentence (top 1)
most_relevant_index = np.argmax(cosine_similarities)
print(most_relevant_index)
exit(0)
most_relevant_sentence = sentences[most_relevant_index]

# Step 6: Extract the topic from the most relevant sentence
# A simple approach: Extract everything after 'about' (if the sentence has it)
# In this case, we assume the topic comes after a phrase like 'about' or 'in' for simplicity.
if "about" in most_relevant_sentence:
    topic = most_relevant_sentence.split("about")[-1].strip()
else:
    topic = most_relevant_sentence

# Step 7: Define the template for LLM content generation
template = f"""
Please write a detailed 300-word document on the following topic:
'{topic}'
The document should explain the concept in an easy-to-understand manner, covering its significance, main principles, and common applications.
"""

# Step 8: Set up OpenAI API for document generation (adjust with your own API key)
openai.api_key = "YOUR_OPENAI_API_KEY"  # Replace with your OpenAI API key

# Step 9: Use OpenAI to generate content based on the topic
response = openai.Completion.create(
    engine="text-davinci-003",  # Choose the appropriate model
    prompt=template,
    max_tokens=500,  # To generate about 300 words
    temperature=0.7  # Controls the creativity of the output
)

# Step 10: Extract and display the generated document
generated_document = response.choices[0].text.strip()

# Step 11: Output the result
print(f"Generated Document on '{topic}':\n")
print(generated_document)


/Users/kaps/Developer/thinkOpenAi/.venv/lib/python3.11/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


IndexError: list index out of range

## 🏁 Wrap-Up

You learned:
- Document chunking & overlap strategies
- Dense embeddings: E5, BGE, MiniLM
- Vector databases: Chroma, FAISS, Qdrant, Weaviate
- Similarity search (cosine, dot-product)
- Sparse retrieval: BM25, TF-IDF
- Cross-encoder reranking
- Context injection & templating


## **Next Topic 3.1: Evaluation using RAGAS, recall@k, grounding** 📊
